# EX_10 — LangGraph y flujos (ejercicios)

**Notebook de referencia:** `notebook/10_LangGraph_Flujos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Estado TypedDict

Define un `TypedDict` de estado con al menos: `question: str`, `answer: str`, `step_count: int`.


In [1]:
from typing import TypedDict

# TODO: add fields question: str, answer: str, step_count: int
class QAState(TypedDict):
    question: str
    answer: str
    step_count: int

## Actividad 2 — Dos nodos

Esquematiza (pseudocódigo) nodos `retrieve` y `generate` que incrementen `step_count`. No hace falta ejecutar LangGraph si aún no está importado en el entorno.


In [2]:
# Pseudocódigo para los nodos de recuperación y generación

def retrieve(state):
    # 1. Leemos el paso actual (si no existe, asumimos que es 0)
    current_step = state.get("step_count", 0)
    
    # 2. Aquí iría la lógica real de buscar en el Vectorstore
    print(f"Buscando documentos para la pregunta: {state.get('question')}")
    
    # 3. Retornamos SOLO lo que queremos actualizar en el estado central
    return {"step_count": current_step + 1}


def generate(state):
    # 1. Leemos el paso actual
    current_step = state.get("step_count", 0)
    
    # 2. Aquí iría la lógica real de pasarle el contexto y la pregunta al LLM
    print("Generando la respuesta con el LLM...")
    respuesta_simulada = "Esta es la respuesta basada en los documentos."
    
    # 3. Retornamos la nueva respuesta y sumamos otro paso al contador
    return {
        "answer": respuesta_simulada, 
        "step_count": current_step + 1
    }

## Actividad 3 — Condicional

Describe en markdown cuándo enrutarías a un nodo `human_review` (p. ej. si `confidence < 0.5`).


### Criterios de enrutamiento al nodo `human_review`

Enrutaría el flujo de LangGraph hacia un nodo de revisión humana (`human_review`) mediante una arista condicional en los siguientes escenarios:

* **Nivel de confianza bajo (Incertidumbre):** Si el LLM genera una respuesta pero el sistema detecta que su nivel de certeza es bajo (p. ej., `confidence < 0.5`). Esto evita que la IA "alucine" o entregue información incorrecta al usuario final.
* **Ejecución de acciones críticas:** Si el siguiente paso del agente implica una acción irreversible o de alto impacto comercial (p. ej., llamar a las herramientas `enviar_correo_masivo`, `ejecutar_reembolso_bancario` o `borrar_registro`). El humano debe dar el "OK" final.
* **Bucle de errores (Max Steps):** Si el contador de pasos (`step_count`) alcanza el límite máximo permitido. Esto indica que el agente está atascado intentando usar una herramienta y necesita que un humano intervenga para desatascar el proceso.
* **Detección de contenido sensible (Moderación):** Si la pregunta del usuario o la respuesta generada activan alertas de seguridad (p. ej., temas médicos, legales o palabras tóxicas).